In [3]:
# CELL 1 — Imports, locked config, dataset loader
import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import os
import json
import glob

# ---- Locked Phase 1 config ----
PATCH_SIZE = 15
PCA_VARIANCE_THRESHOLD = 0.99
K_FLOOR = 10
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.10, 0.10, 0.80
RANDOM_STATE_SPLIT = 1000  # locked split seed, distinct from model seeds [42,43,44]

# ---- Base directory (your local dataset root) ----
BASE_DIR = r"C:\Users\aipmu\Downloads\Remote-Sensing-Datasets\HSI Datasets"

# ---- Dataset registry ----
# Filenames below follow the standard eecn/Hyperspectral-Classification naming
# convention. If your folders use different filenames, the loader in this cell
# will list the folder contents so you can correct the name quickly.
DATASET_REGISTRY = {
    "IndianPines": {
        "data_path": os.path.join(BASE_DIR, "Indian_Pines", "Indian_pines_corrected.mat"),
        "data_key": "indian_pines_corrected",
        "gt_path": os.path.join(BASE_DIR, "Indian_Pines", "Indian_pines_gt.mat"),
        "gt_key": "indian_pines_gt",
        "padding": "zero", "fidelity_checkpoint": True,
    },
    "PaviaUniversity": {
        "data_path": os.path.join(BASE_DIR, "Pavia University", "PaviaU.mat"),
        "data_key": "paviaU",
        "gt_path": os.path.join(BASE_DIR, "Pavia University", "PaviaU_gt.mat"),
        "gt_key": "paviaU_gt",
        "padding": "zero", "fidelity_checkpoint": True,
    },
    "Salinas": {
        "data_path": os.path.join(BASE_DIR, "Salinas Scene", "Salinas_corrected.mat"),
        "data_key": "salinas_corrected",
        "gt_path": os.path.join(BASE_DIR, "Salinas Scene", "Salinas_gt.mat"),
        "gt_key": "salinas_gt",
        "padding": "reflect", "fidelity_checkpoint": False,
    },
    "KSC": {
        "data_path": os.path.join(BASE_DIR, "Kennedy Space Center (KSC)", "KSC.mat"),
        "data_key": "KSC",
        "gt_path": os.path.join(BASE_DIR, "Kennedy Space Center (KSC)", "KSC_gt.mat"),
        "gt_key": "KSC_gt",
        "padding": "reflect", "fidelity_checkpoint": False,
    },
    "Botswana": {
        "data_path": os.path.join(BASE_DIR, "Botswana", "Botswana.mat"),
        "data_key": "Botswana",
        "gt_path": os.path.join(BASE_DIR, "Botswana", "Botswana_gt.mat"),
        "gt_key": "Botswana_gt",
        "padding": "reflect", "fidelity_checkpoint": False,
    },
    # Houston 2013 folder exists at:
    #   os.path.join(BASE_DIR, "Houston 13")
    # but is intentionally NOT registered here — it uses the official DFC2013
    # train/test partition, not the stratified_pixel_split() in Cell 2.
    # Needs its own loader once we confirm the file layout inside that folder.
}

def load_mat_dataset(data_path, data_key, gt_path, gt_key):
    if not os.path.exists(data_path) or not os.path.exists(gt_path):
        folder = os.path.dirname(data_path)
        print(f"  [FILE NOT FOUND] Expected: {data_path}")
        if os.path.isdir(folder):
            print(f"  Actual contents of '{folder}':")
            for f in glob.glob(os.path.join(folder, "*")):
                print(f"    - {os.path.basename(f)}")
        else:
            print(f"  Folder does not exist: {folder}")
        raise FileNotFoundError(f"Missing file: {data_path} or {gt_path}")
    X = sio.loadmat(data_path)[data_key].astype(np.float32)   # (H, W, Bands)
    y = sio.loadmat(gt_path)[gt_key].astype(np.int64)         # (H, W), 0 = background
    print(f"  loaded X{X.shape}, y{y.shape}, classes={np.unique(y)}")
    return X, y

print("Cell 1 loaded: config + dataset registry ready.")
print(f"Base directory: {BASE_DIR}")
print(f"Registered datasets: {list(DATASET_REGISTRY.keys())} (+ Houston, handled separately)")

Cell 1 loaded: config + dataset registry ready.
Base directory: C:\Users\aipmu\Downloads\Remote-Sensing-Datasets\HSI Datasets
Registered datasets: ['IndianPines', 'PaviaUniversity', 'Salinas', 'KSC', 'Botswana'] (+ Houston, handled separately)


In [4]:
# CELL 2 — Stratified per-class split, BEFORE any statistic is fit
# Order enforced: split -> (fit stats on train only, done in Cell 3) -> patch (Cell 4)

def stratified_pixel_split(y, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                            random_state=RANDOM_STATE_SPLIT):
    """
    Returns three (row, col) coordinate arrays: train, val, test.
    Background pixels (label 0) are excluded entirely.
    """
    coords = np.argwhere(y > 0)                      # (N, 2) -> row, col
    labels = y[coords[:, 0], coords[:, 1]]

    # flag classes too small to stratify safely
    unique, counts = np.unique(labels, return_counts=True)
    tiny_classes = unique[counts < 10]
    if len(tiny_classes) > 0:
        print(f"  [WARNING] classes with <10 labeled pixels: {tiny_classes.tolist()} "
              f"— stratified split may be unstable for these; inspect before proceeding.")

    train_coords, temp_coords, train_labels, temp_labels = train_test_split(
        coords, labels, train_size=train_ratio, stratify=labels,
        random_state=random_state,
    )
    # remaining split into val/test proportional to the original ratios
    remaining_val_fraction = val_ratio / (val_ratio + (1 - train_ratio - val_ratio))
    val_coords, test_coords = train_test_split(
        temp_coords, train_size=remaining_val_fraction, stratify=temp_labels,
        random_state=random_state,
    )
    print(f"  split sizes -> train: {len(train_coords)}, val: {len(val_coords)}, "
          f"test: {len(test_coords)}")
    return train_coords, val_coords, test_coords

print("Cell 2 loaded: stratified_pixel_split() ready.")

Cell 2 loaded: stratified_pixel_split() ready.


In [5]:
# CELL 3 — Normalization + PCA, fit strictly on the training split

def fit_normalization_and_pca(X, train_coords, variance_threshold=PCA_VARIANCE_THRESHOLD,
                               k_floor=K_FLOOR):
    """
    Fits min-max stats and PCA using ONLY the training pixel spectra.
    Returns fitted (band_min, band_max, pca_model, k_used, k_flagged).
    """
    train_spectra = X[train_coords[:, 0], train_coords[:, 1], :]   # (N_train, Bands)

    band_min = train_spectra.min(axis=0)
    band_max = train_spectra.max(axis=0)
    eps = 1e-6
    train_norm = (train_spectra - band_min) / (band_max - band_min + eps)

    pca_full = PCA(n_components=min(train_norm.shape[0], train_norm.shape[1]))
    pca_full.fit(train_norm)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    k_natural = int(np.searchsorted(cumvar, variance_threshold) + 1)

    k_flagged = k_natural < k_floor
    k_used = max(k_natural, k_floor)
    if k_flagged:
        print(f"  [FLAG] natural k={k_natural} is below floor={k_floor}. "
              f"Using k={k_used}, but INSPECT this dataset before proceeding to training.")
    else:
        print(f"  k={k_used} components reach {variance_threshold*100:.0f}% variance "
              f"(natural k={k_natural})")

    pca_final = PCA(n_components=k_used)
    pca_final.fit(train_norm)

    return {
        "band_min": band_min, "band_max": band_max, "eps": eps,
        "pca_model": pca_final, "k_used": k_used, "k_natural": k_natural,
        "k_flagged": k_flagged,
    }

def apply_normalization_and_pca(X, stats):
    band_min, band_max, eps = stats["band_min"], stats["band_max"], stats["eps"]
    H, W, B = X.shape
    X_norm = (X - band_min) / (band_max - band_min + eps)
    X_flat = X_norm.reshape(-1, B)
    X_pca = stats["pca_model"].transform(X_flat)
    return X_pca.reshape(H, W, stats["k_used"])

print("Cell 3 loaded: fit_normalization_and_pca() / apply_normalization_and_pca() ready.")

Cell 3 loaded: fit_normalization_and_pca() / apply_normalization_and_pca() ready.


In [6]:
# CELL 4 — 15x15 patch extraction + tokenization
# Augmentation (if any) must be applied to raw patches HERE, before tokens are
# flattened, per the locked ordering requirement (protects positional encoding).

def extract_patches(X_pca, coords, patch_size=PATCH_SIZE, padding="zero"):
    """
    X_pca: (H, W, k) already PCA-transformed.
    coords: (N, 2) row,col pixel centers.
    Returns patches: (N, patch_size, patch_size, k)
    """
    half = patch_size // 2
    pad_mode = "constant" if padding == "zero" else "reflect"
    pad_kwargs = {"constant_values": 0} if padding == "zero" else {}

    X_padded = np.pad(
        X_pca, ((half, half), (half, half), (0, 0)), mode=pad_mode, **pad_kwargs
    )
    patches = np.empty((len(coords), patch_size, patch_size, X_pca.shape[2]),
                        dtype=X_pca.dtype)
    for i, (r, c) in enumerate(coords):
        r_p, c_p = r + half, c + half   # offset into padded array
        patches[i] = X_padded[r_p - half:r_p + half + 1, c_p - half:c_p + half + 1, :]
    return patches

def tokenize(patches):
    """(N, 15, 15, k) -> (N, 225, k)"""
    n, p, _, k = patches.shape
    return patches.reshape(n, p * p, k)

def augment_patch_grid(patch, rng):
    """
    Applied to a single (15, 15, k) patch BEFORE tokenization.
    Flip/rotate on the spatial grid; noise added to raw normalized values
    (call this only on data already in normalized-band space, pre-PCA, if
    noise augmentation is desired per the locked protocol — see note below).
    """
    if rng.random() < 0.5:
        patch = np.flip(patch, axis=0)
    if rng.random() < 0.5:
        patch = np.flip(patch, axis=1)
    k_rot = rng.integers(0, 4)
    patch = np.rot90(patch, k=k_rot, axes=(0, 1))
    return patch

print("Cell 4 loaded: extract_patches() / tokenize() / augment_patch_grid() ready.")
print("NOTE: noise augmentation (std 0.01-0.03) must be injected in RAW min-max-normalized")
print("band space, BEFORE PCA transform — apply it in the pipeline prior to Cell 3's PCA step")
print("for any augmented training variant, not on the PCA-transformed patches here.")

Cell 4 loaded: extract_patches() / tokenize() / augment_patch_grid() ready.
NOTE: noise augmentation (std 0.01-0.03) must be injected in RAW min-max-normalized
band space, BEFORE PCA transform — apply it in the pipeline prior to Cell 3's PCA step
for any augmented training variant, not on the PCA-transformed patches here.


In [7]:
# CELL 5 — Full Phase 1 run: all datasets, report real n_train/k to replace
# Phase 0's placeholder values (n_train=8000, k=20)

phase1_report = {}

for name, cfg in DATASET_REGISTRY.items():
    print(f"\n{'='*60}\nDataset: {name}\n{'='*60}")
    try:
        X, y = load_mat_dataset(cfg["data_path"], cfg["data_key"], cfg["gt_path"], cfg["gt_key"])
        train_c, val_c, test_c = stratified_pixel_split(y)
        stats = fit_normalization_and_pca(X, train_c)
        X_pca = apply_normalization_and_pca(X, stats)

        train_patches = extract_patches(X_pca, train_c, padding=cfg["padding"])
        val_patches = extract_patches(X_pca, val_c, padding=cfg["padding"])
        test_patches = extract_patches(X_pca, test_c, padding=cfg["padding"])

        train_tokens = tokenize(train_patches)
        val_tokens = tokenize(val_patches)
        test_tokens = tokenize(test_patches)

        n_classes = len(np.unique(y)) - 1  # exclude background

        phase1_report[name] = {
            "n_train": len(train_c), "n_val": len(val_c), "n_test": len(test_c),
            "k_used": stats["k_used"], "k_natural": stats["k_natural"],
            "k_flagged": bool(stats["k_flagged"]),
            "n_classes": n_classes, "padding": cfg["padding"],
            "fidelity_checkpoint": cfg["fidelity_checkpoint"],
        }
        print(f"  READY: n_train={len(train_c)}, k={stats['k_used']}, "
              f"n_classes={n_classes}, tokens shape={train_tokens.shape}")

        # Save to disk for Phase 2
        os.makedirs("preprocessed", exist_ok=True)
        np.savez(f"preprocessed/{name}.npz",
                  train_tokens=train_tokens, train_labels=y[train_c[:,0], train_c[:,1]],
                  val_tokens=val_tokens, val_labels=y[val_c[:,0], val_c[:,1]],
                  test_tokens=test_tokens, test_labels=y[test_c[:,0], test_c[:,1]])

    except FileNotFoundError as e:
        print(f"  [SKIP] {e} — update DATASET_REGISTRY paths in Cell 1.")
        phase1_report[name] = {"status": "file_not_found"}

print(f"\n{'='*60}\nPHASE 1 SUMMARY — real values to replace Phase 0 placeholders\n{'='*60}")
for name, r in phase1_report.items():
    print(f"{name}: {r}")

with open("phase1_report.json", "w") as f:
    json.dump(phase1_report, f, indent=2)
print("\nSaved phase1_report.json — use each dataset's real n_train/k in the Phase 0")
print("extrapolation formula to replace the placeholder 8000/20 estimate.")
print("\nHouston 2013 (official DFC2013 partition) is NOT included above — it needs its own")
print("loader using the official train/test masks rather than stratified_pixel_split().")


Dataset: IndianPines
  loaded X(145, 145, 200), y(145, 145), classes=[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]
  split sizes -> train: 1024, val: 1025, test: 8200
  k=31 components reach 99% variance (natural k=31)
  READY: n_train=1024, k=31, n_classes=16, tokens shape=(1024, 225, 31)

Dataset: PaviaUniversity
  loaded X(610, 340, 103), y(610, 340), classes=[0 1 2 3 4 5 6 7 8 9]
  split sizes -> train: 4277, val: 4277, test: 34222
  [FLAG] natural k=4 is below floor=10. Using k=10, but INSPECT this dataset before proceeding to training.
  READY: n_train=4277, k=10, n_classes=9, tokens shape=(4277, 225, 10)

Dataset: Salinas
  loaded X(512, 217, 204), y(512, 217), classes=[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]
  split sizes -> train: 5412, val: 5413, test: 43304
  [FLAG] natural k=4 is below floor=10. Using k=10, but INSPECT this dataset before proceeding to training.
  READY: n_train=5412, k=10, n_classes=16, tokens shape=(5412, 225, 10)

Dataset: KSC
  loade

In [ ]:
# CELL 6 — Same diagnostic, extended to KSC and Botswana, with a fuller variance curve
for name in ["KSC", "Botswana"]:
    cfg = DATASET_REGISTRY[name]
    X, y = load_mat_dataset(cfg["data_path"], cfg["data_key"], cfg["gt_path"], cfg["gt_key"])
    train_c, _, _ = stratified_pixel_split(y)
    train_spectra = X[train_c[:,0], train_c[:,1], :]

    band_min = train_spectra.min(axis=0)
    band_max = train_spectra.max(axis=0)
    print(f"\n{name}:")
    print(f"  bands: {train_spectra.shape[1]}")
    print(f"  raw value range: [{train_spectra.min():.2f}, {train_spectra.max():.2f}]")
    p99 = np.percentile(train_spectra, 99, axis=0)
    print(f"  outlier check (true_max / p99_max): {band_max.max() / p99.max():.2f}")

    eps = 1e-6
    train_norm = (train_spectra - band_min) / (band_max - band_min + eps)

    # check for constant/near-zero-variance bands, which can artificially
    # concentrate PCA variance in a few components
    band_std = train_norm.std(axis=0)
    dead_bands = np.sum(band_std < 1e-4)
    print(f"  near-constant bands (std < 1e-4): {dead_bands} / {train_spectra.shape[1]}")

    pca_check = PCA(n_components=20).fit(train_norm)
    cumvar = np.cumsum(pca_check.explained_variance_ratio_)
    print(f"  cumulative variance, components 1-20:")
    print(f"  {np.round(cumvar, 4)}")

  loaded X(512, 614, 176), y(512, 614), classes=[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
  split sizes -> train: 521, val: 521, test: 4169

KSC:
  bands: 176
  raw value range: [0.00, 65535.00]
  outlier check (true_max / p99_max): 170.49
  near-constant bands (std < 1e-4): 0 / 176
  cumulative variance, components 1-20:
  [0.7396 0.9406 0.9872 0.991  0.9936 0.9947 0.9957 0.9963 0.9968 0.9971
 0.9975 0.9979 0.9981 0.9982 0.9983 0.9984 0.9985 0.9986 0.9987 0.9988]
  loaded X(1476, 256, 145), y(1476, 256), classes=[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
  split sizes -> train: 324, val: 324, test: 2600

Botswana:
  bands: 145
  raw value range: [0.00, 8274.00]
  outlier check (true_max / p99_max): 1.24
  near-constant bands (std < 1e-4): 0 / 145
  cumulative variance, components 1-20:
  [0.8947 0.978  0.9882 0.9916 0.993  0.994  0.9945 0.9949 0.9953 0.9955
 0.9958 0.996  0.9962 0.9964 0.9966 0.9967 0.9969 0.997  0.9972 0.9973]


In [ ]:
# CELL 7 — Inspect Houston 2013 .mat file contents before writing its loader
import scipy.io as sio

houston_dir = os.path.join(BASE_DIR, "Houston 13")  # adjust if the folder name differs
data_path = os.path.join(houston_dir, "houston13_data.mat")
gt_path = os.path.join(houston_dir, "houston13_gt.mat")

for label, path in [("DATA", data_path), ("GT", gt_path)]:
    print(f"\n{label}: {path}")
    mat = sio.loadmat(path)
    for key, val in mat.items():
        if key.startswith("__"):
            continue
        shape = getattr(val, "shape", None)
        dtype = getattr(val, "dtype", None)
        print(f"  key='{key}'  shape={shape}  dtype={dtype}")
        if shape is not None and len(shape) <= 2 and np.prod(shape) < 50:
            print(f"    values: {val}")


DATA: C:\Users\aipmu\Downloads\Remote-Sensing-Datasets\HSI Datasets\Houston 13\houston13_data.mat
  key='data'  shape=(349, 1905, 144)  dtype=uint16

GT: C:\Users\aipmu\Downloads\Remote-Sensing-Datasets\HSI Datasets\Houston 13\houston13_gt.mat
  key='gt'  shape=(349, 1905)  dtype=uint8


In [ ]:
# CELL 8 — KSC-specific fix: handle sensor-saturation outliers before PCA
cfg = DATASET_REGISTRY["KSC"]
X, y = load_mat_dataset(cfg["data_path"], cfg["data_key"], cfg["gt_path"], cfg["gt_key"])
train_c, _, _ = stratified_pixel_split(y)
train_spectra = X[train_c[:,0], train_c[:,1], :]

SATURATION_VALUE = 65535
saturated_mask = (train_spectra >= SATURATION_VALUE)
print(f"Saturated entries: {saturated_mask.sum()} / {train_spectra.size} "
      f"({100*saturated_mask.sum()/train_spectra.size:.3f}%)")
print(f"Pixels with at least one saturated band: {saturated_mask.any(axis=1).sum()} / {len(train_spectra)}")

# Use 99.9th percentile per band as the effective max instead of the true max,
# clipping saturated values down to that ceiling before min-max scaling.
band_min = train_spectra.min(axis=0)
band_max_robust = np.percentile(train_spectra, 99.9, axis=0)
train_clipped = np.clip(train_spectra, band_min, band_max_robust)
eps = 1e-6
train_norm_fixed = (train_clipped - band_min) / (band_max_robust - band_min + eps)

pca_fixed = PCA(n_components=20).fit(train_norm_fixed)
cumvar_fixed = np.cumsum(pca_fixed.explained_variance_ratio_)
print(f"KSC cumulative variance AFTER saturation fix, components 1-20:")
print(np.round(cumvar_fixed, 4))

  loaded X(512, 614, 176), y(512, 614), classes=[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
  split sizes -> train: 521, val: 521, test: 4169
Saturated entries: 5 / 91696 (0.005%)
Pixels with at least one saturated band: 5 / 521
KSC cumulative variance AFTER saturation fix, components 1-20:
[0.7395 0.9404 0.9872 0.991  0.9936 0.9947 0.9957 0.9963 0.9968 0.9971
 0.9975 0.9979 0.998  0.9982 0.9983 0.9984 0.9985 0.9986 0.9987 0.9988]


In [ ]:
# CELL 9 — Fix Houston's axis order to match the rest of the pipeline (H, W, bands)
hsi_hwc = np.transpose(hsi, (1, 2, 0))   # (144,349,1905) -> (349,1905,144)
print(f"Reordered HSI shape: {hsi_hwc.shape}")  # expect (349, 1905, 144)

train_label_arr = np.asarray(train_label)   # (349, 1905) presumably — verify below
test_label_arr = np.asarray(test_label)
print(f"train_label shape: {train_label_arr.shape}, test_label shape: {test_label_arr.shape}")
assert train_label_arr.shape == hsi_hwc.shape[:2], \
    "Label shape doesn't match transposed HSI spatial dims — check orientation before proceeding."

Reordered HSI shape: (349, 1905, 144)
train_label shape: (349, 1905), test_label shape: (349, 1905)


In [ ]:
# CELL 10 — Houston 2013: full pipeline using the official partition
train_labels_at_coords = train_label_arr[train_coords[:,0], train_coords[:,1]]

houston_train_c, houston_val_c = train_test_split(
    train_coords, train_size=0.9, stratify=train_labels_at_coords,
    random_state=RANDOM_STATE_SPLIT,
)
houston_test_c = test_coords  # official test partition, untouched

print(f"Houston final split -> train: {len(houston_train_c)}, "
      f"val: {len(houston_val_c)}, test: {len(houston_test_c)}")

# fit normalization + PCA on the official training portion only
houston_stats = fit_normalization_and_pca(hsi_hwc, houston_train_c)
houston_pca = apply_normalization_and_pca(hsi_hwc, houston_stats)

# reflection padding, per the locked protocol for the full 15-class scene
houston_padding = "reflect"

houston_train_patches = extract_patches(houston_pca, houston_train_c, padding=houston_padding)
houston_val_patches = extract_patches(houston_pca, houston_val_c, padding=houston_padding)
houston_test_patches = extract_patches(houston_pca, houston_test_c, padding=houston_padding)

houston_train_tokens = tokenize(houston_train_patches)
houston_val_tokens = tokenize(houston_val_patches)
houston_test_tokens = tokenize(houston_test_patches)

houston_n_classes = len(np.unique(train_labels_at_coords))

phase1_report["Houston2013_Full15"] = {
    "n_train": len(houston_train_c), "n_val": len(houston_val_c), "n_test": len(houston_test_c),
    "k_used": houston_stats["k_used"], "k_natural": houston_stats["k_natural"],
    "k_flagged": bool(houston_stats["k_flagged"]),
    "n_classes": houston_n_classes, "padding": houston_padding,
    "fidelity_checkpoint": False,
    "split_type": "official_DFC2013_partition",
}
print(f"READY: n_train={len(houston_train_c)}, k={houston_stats['k_used']}, "
      f"n_classes={houston_n_classes}, tokens shape={houston_train_tokens.shape}")

os.makedirs("preprocessed", exist_ok=True)
np.savez(
    "preprocessed/Houston2013_Full15.npz",
    train_tokens=houston_train_tokens,
    train_labels=train_label_arr[houston_train_c[:, 0], houston_train_c[:, 1]],
    val_tokens=houston_val_tokens,
    val_labels=train_label_arr[houston_val_c[:, 0], houston_val_c[:, 1]],
    test_tokens=houston_test_tokens,
    test_labels=test_label_arr[houston_test_c[:, 0], houston_test_c[:, 1]],
)

with open("phase1_report.json", "w") as f:
    json.dump(phase1_report, f, indent=2)
print("\nUpdated phase1_report.json with Houston2013_Full15.")

Houston final split -> train: 2535, val: 282, test: 12182
  [FLAG] natural k=4 is below floor=10. Using k=10, but INSPECT this dataset before proceeding to training.
READY: n_train=2535, k=10, n_classes=15, tokens shape=(2535, 225, 10)

Updated phase1_report.json with Houston2013_Full15.


In [ ]:
# CELL 11 — Diagnostic: why did Houston also hit the k-floor?
houston_train_spectra = hsi_hwc[houston_train_c[:,0], houston_train_c[:,1], :]

band_min = houston_train_spectra.min(axis=0)
band_max = houston_train_spectra.max(axis=0)
print(f"Houston2013_Full15:")
print(f"  bands: {houston_train_spectra.shape[1]}")
print(f"  raw value range: [{houston_train_spectra.min():.2f}, {houston_train_spectra.max():.2f}]")

p99 = np.percentile(houston_train_spectra, 99, axis=0)
print(f"  outlier check (true_max / p99_max): {band_max.max() / p99.max():.2f}")

eps = 1e-6
train_norm = (houston_train_spectra - band_min) / (band_max - band_min + eps)
band_std = train_norm.std(axis=0)
dead_bands = np.sum(band_std < 1e-4)
print(f"  near-constant bands (std < 1e-4): {dead_bands} / {houston_train_spectra.shape[1]}")

pca_check = PCA(n_components=20).fit(train_norm)
cumvar = np.cumsum(pca_check.explained_variance_ratio_)
print(f"  cumulative variance, components 1-20:")
print(f"  {np.round(cumvar, 4)}")

Houston2013_Full15:
  bands: 144
  raw value range: [0.00, 43352.00]
  outlier check (true_max / p99_max): 1.34
  near-constant bands (std < 1e-4): 0 / 144
  cumulative variance, components 1-20:
  [0.7337 0.9698 0.9857 0.9939 0.9959 0.9973 0.9981 0.9986 0.9989 0.9991
 0.9993 0.9994 0.9995 0.9995 0.9996 0.9996 0.9997 0.9997 0.9997 0.9997]


In [ ]:
# CELL 12 — Check whether test coordinates were saved anywhere, and regenerate if not
import os

# First: confirm they're genuinely absent from the saved .npz files
print("Checking preprocessed/*.npz for coordinate data...\n")
for name in ["IndianPines", "PaviaUniversity", "Salinas", "KSC", "Botswana"]:
    path = f"preprocessed/{name}.npz"
    if os.path.exists(path):
        d = np.load(path)
        print(f"{name}: keys = {list(d.keys())}")
    else:
        print(f"{name}: file not found at {path}")

houston_path = "preprocessed/Houston2013_Full15.npz"
if os.path.exists(houston_path):
    d = np.load(houston_path)
    print(f"Houston2013_Full15: keys = {list(d.keys())}")

print("\n" + "="*60)
print("If 'test_coords' or similar does not appear above, they were never saved.")
print("Regenerating deterministically using the locked split seed (1000)...")
print("="*60)

# Regenerate test coordinates for the five standard-split datasets
regenerated_coords = {}
for name, cfg in DATASET_REGISTRY.items():
    print(f"\n{name}:")
    X, y = load_mat_dataset(cfg["data_path"], cfg["data_key"], cfg["gt_path"], cfg["gt_key"])
    train_c, val_c, test_c = stratified_pixel_split(y)  # same locked seed (1000) as original run
    regenerated_coords[name] = {
        "test_coords": test_c,
        "scene_shape": y.shape,  # (H, W)
    }
    print(f"  scene_shape={y.shape}, test_coords shape={test_c.shape}")

    # Sanity check: does the regenerated test set size match what's already saved?
    saved = np.load(f"preprocessed/{name}.npz")
    match = len(test_c) == len(saved["test_labels"])
    print(f"  matches saved test set size ({len(saved['test_labels'])})? {'YES' if match else 'NO — MISMATCH, investigate before trusting this'}")

# Save these separately so we don't have to regenerate every time
np.savez("preprocessed/test_coordinates.npz",
          **{f"{name}_coords": regenerated_coords[name]["test_coords"] for name in regenerated_coords},
          **{f"{name}_shape": regenerated_coords[name]["scene_shape"] for name in regenerated_coords})
print("\nSaved preprocessed/test_coordinates.npz for reuse in classification maps.")

Checking preprocessed/*.npz for coordinate data...

IndianPines: keys = ['train_tokens', 'train_labels', 'val_tokens', 'val_labels', 'test_tokens', 'test_labels']
PaviaUniversity: keys = ['train_tokens', 'train_labels', 'val_tokens', 'val_labels', 'test_tokens', 'test_labels']
Salinas: keys = ['train_tokens', 'train_labels', 'val_tokens', 'val_labels', 'test_tokens', 'test_labels']
KSC: keys = ['train_tokens', 'train_labels', 'val_tokens', 'val_labels', 'test_tokens', 'test_labels']
Botswana: keys = ['train_tokens', 'train_labels', 'val_tokens', 'val_labels', 'test_tokens', 'test_labels']
Houston2013_Full15: keys = ['train_tokens', 'train_labels', 'val_tokens', 'val_labels', 'test_tokens', 'test_labels']

If 'test_coords' or similar does not appear above, they were never saved.
Regenerating deterministically using the locked split seed (1000)...

IndianPines:
  loaded X(145, 145, 200), y(145, 145), classes=[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16]
  split sizes -> train: 102